# Unicycle leftovers

Four networks, one recipe. Train/val is the first 75% / last 25% of the seven training collections. Test is figure 8, diagonal lawnmower, slalom (held out whole).

Train on **5 s** windows. The checkpoint is the lowest **5 s** xyyaw on val. Commands are commanded `(v, ω)`.

Checkpoints: `runs/unicycle/<family>/seed_k/{best.pt,last.pt}`. Training resumes `last.pt` if it exists. After each family finishes, the notebook overlays a full open-loop rollout on every held-out test collection.

In [ ]:
SEED = 0  # change to start or resume another copy; eval uses seeds 0–4
DEVICE = "mps"

## Dataset

Left vs right commanded turns, and the `(v, ω)` the robot was asked to run.

In [ ]:
from unicycle.experiments.data import load_split, turn_counts
from unicycle.experiments.viz import plot_command_heatmap, plot_split_balance

train, val, test = load_split()
for name, logs in (("train", train), ("val", val), ("test", test)):
    print(name, turn_counts(logs))
plot_split_balance(train, val, test)
plot_command_heatmap(train, title="train")
plot_command_heatmap(test, title="test")

## Visualizations

Writes `experiments/visualizations/stills` and `gifs` for every train, val, and test collection.

In [ ]:
from unicycle.experiments.viz import write_collections

write_collections(train, val, test)

## NaiveUnicycle

MLP on the 7-D world state. No SE(2) quotient.

In [ ]:
from quadrotor.geometry import train_device
from unicycle.experiments.models import NAIVE, QUOTIENT, QUOTIENT_AUG, QUOTIENT_SYM
from unicycle.experiments.train.lib import train_family
from unicycle.experiments.viz import plot_test_rollouts

device = train_device(DEVICE)
model = train_family(NAIVE, SEED, device=device)
plot_test_rollouts(model, test, NAIVE)

## QuotientUnicycleNet

Same leftover, input is `(v, ω, v_cmd, ω_cmd)`.

In [ ]:
model = train_family(QUOTIENT, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT)

## QuotientUnicycleNet + data augmentation

Same quotient net. Training data is concatenated with its left–right mirror.

In [ ]:
model = train_family(QUOTIENT_AUG, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT_AUG)

## QuotientSymUnicycleNet

Quotient leftover. `Orbit` evaluates both Z₂ elements (identity and left–right: wheels swap, ω flips, v stays) and Reynolds-averages.

In [ ]:
model = train_family(QUOTIENT_SYM, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT_SYM)